# m17 lab · Impedance (current-based) localization

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Impedance localization drives small high-frequency currents between skin patches and reads the potential at every catheter electrode. You compute the field of a patch pair, see why a uniform-field map stretches positions toward the patches, map a 3D grid, correct distances with electrodes of known spacing, see how a change of conductivity stretches apparent distance, and turn voltage noise into position noise.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A patch pair and its nonlinearity | eq-m17-pair, eq-m17-axis |
| B | Three pairs and a linear map | eq-m17-linear |
| C | Field scaling from known electrode spacing | eq-m17-metric |
| D | Conductivity changes stretch distance | eq-m17-interface |
| E | Precision | eq-m17-precision |

Lengths in mm; potentials in relative units with I/(4 pi sigma) = 1. The torso is an unbounded homogeneous medium except in part D; patch positions, conductivities and noise are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import impedance as Z

## Part A · A patch pair and its nonlinearity

Current $I$ enters at one patch and leaves at the other. In a homogeneous medium the potential is
$k(1/|\mathbf r - \mathbf a| - 1/|\mathbf r - \mathbf b|)$ with $k = I/(4\pi\sigma)$ (eq-m17-pair). Along the axis
between patches at $\pm d$ it is $2kx/(d^2 - x^2)$, so dividing by the slope at the center reports positions too far
out by the fraction $x^2/(d^2 - x^2)$ (eq-m17-axis).

In [ ]:
d = 200.0
x = np.linspace(-100, 100, 201)
rep = Z.linear_estimate(Z.axis_potential(x, d), d)
plt.plot(x, x, c="gray", label="truth"); plt.plot(x, rep, label="uniform-field estimate")
plt.xlabel("true position (mm)"); plt.ylabel("reported position (mm)"); plt.legend(); plt.show()
for xi in (20.0, 50.0, 80.0):
    r = Z.linear_estimate(Z.axis_potential(xi, d), d)
    print(f"x = {xi:4.0f} mm: reported {r:6.2f} mm, stretch {100 * (r - xi) / xi:5.2f}% (formula {100 * xi**2 / (d**2 - xi**2):5.2f}%)")

In [ ]:
# Checks
r = Z.linear_estimate(Z.axis_potential(50.0, d), d)
assert abs((r - 50) / 50 - 50**2 / (d**2 - 50**2)) < 1e-12

## Part B · Three pairs and a linear map

Three orthogonal pairs, driven at different frequencies, give three potentials at each electrode. A linear map
$\hat x_i = \phi_i/E_0$ (eq-m17-linear) is exact only at the center. Map a 3D grid and measure the error against the
grid's size relative to $d$.

In [ ]:
P = Z.patch_pairs(d)
for w in (20.0, 40.0, 60.0, 80.0):
    g = np.linspace(-w, w, 9)
    pts = np.stack(np.meshgrid(g, g, g), axis=-1).reshape(-1, 3)
    est = Z.linear_estimate(Z.potentials(pts, P), d)
    err = np.linalg.norm(est - pts, axis=1)
    print(f"cube half-width {w:3.0f} mm (w/d = {w/d:.2f}): RMS error {np.sqrt(np.mean(err**2)):5.2f} mm, max {err.max():5.2f} mm")

In [ ]:
# Checks
g = np.linspace(-20, 20, 5); pts = np.stack(np.meshgrid(g, g, g), axis=-1).reshape(-1, 3)
assert np.max(np.linalg.norm(Z.linear_estimate(Z.potentials(pts, P), d) - pts, axis=1)) < 1.0

## Part C · Field scaling from known electrode spacing

Electrodes on a catheter have known spacing. Locally the potentials change as $\Delta\boldsymbol\phi = J\Delta\mathbf x$,
so the true distance satisfies $\ell^2 = \Delta\boldsymbol\phi^\top M\,\Delta\boldsymbol\phi$ with $M = (JJ^\top)^{-1}$
(eq-m17-metric). Each electrode pair gives one linear equation in the six entries of $M$. Sweep a catheter through
several orientations near one point, fit $M$, and measure a new pair.

In [ ]:
r0 = np.array([60.0, 10.0, 10.0])
rng = np.random.default_rng(17)
dphis, ls = [], []
for _ in range(12):                                          # 12 orientations of a 4 mm bipole
    u = rng.standard_normal(3); u /= np.linalg.norm(u)
    dphis.append(Z.potentials(r0 + 2 * u, P) - Z.potentials(r0 - 2 * u, P)); ls.append(4.0)
M = Z.fit_metric(dphis, ls)
u = np.array([1.0, 0.0, 0.0])
test = Z.potentials(r0 + 5 * u, P) - Z.potentials(r0 - 5 * u, P)
raw = np.linalg.norm(Z.linear_estimate(Z.potentials(r0 + 5 * u, P), d) - Z.linear_estimate(Z.potentials(r0 - 5 * u, P), d))
print(f"a 10 mm pair: uniform-field map {raw:.2f} mm, field-scaled {Z.metric_distance(test, M):.2f} mm")

In [ ]:
# Checks
assert abs(Z.metric_distance(test, M) - 10.0) < 0.02
assert abs(raw - 10.0) > 0.5

## Part D · Conductivity changes stretch distance

Normal current is continuous across an interface, so where conductivity is lower the field is higher by
$\sigma_{\text{ref}}/\sigma_{\text{local}}$ (eq-m17-interface), and a potential-based system sees distances there as longer.
A 1D layered model: blood, a slab of lower conductivity, blood. The conductivity ratio is illustrative.

In [ ]:
x = np.linspace(0, 30, 301)
for ratio in (1.0, 0.5, 0.3):
    plt.plot(x, Z.layered_positions(x, [0, 10, 20, 30], [1.0, ratio, 1.0]), label=f"middle layer σ ratio {ratio}")
plt.xlabel("true position (mm)"); plt.ylabel("apparent position (mm)"); plt.legend(); plt.show()
app = Z.layered_positions(np.array([10.0, 20.0]), [0, 10, 20, 30], [1.0, 0.5, 1.0])
print("a 10 mm slab at half the conductivity appears", app[1] - app[0], "mm thick")

In [ ]:
# Checks
assert abs((app[1] - app[0]) - 20.0) < 1e-12

## Part E · Precision

Voltage noise $\delta\phi$ becomes position noise $\delta\phi/|E|$ (eq-m17-precision). The field at the center is
$2k/d^2$: patches closer together, or more current, give a stronger field and less position noise, at the price of more
nonlinearity (part A).

In [ ]:
for dd in (150.0, 200.0, 300.0):
    print(f"d = {dd:3.0f} mm: field {Z.center_field(dd):.2e} per mm; noise 1e-6 gives {Z.position_noise(1e-6, Z.center_field(dd)):.3f} mm; "
          f"stretch at 60 mm {100 * 60**2 / (dd**2 - 60**2):.1f}%")

In [ ]:
# Checks
assert Z.position_noise(1e-6, Z.center_field(300.0)) > Z.position_noise(1e-6, Z.center_field(150.0))

## Where to go next

- Put a low-conductivity region (a lung) beside the heart using the m02 finite-difference solver and map the
  distortion.
- Fit a smooth global correction from many catheter positions, each with known spacings.
- m18 combines this with magnetic localization; m19 removes respiration and drift.